In [1]:
import torch
from huggingface_hub import snapshot_download

/home/jamesliu/anaconda3/envs/sglang/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
# thunlp/LLaMA3-Instruct-8B-FR-Spec/freq_32768.pt
def load_token_map(token_map_path: str):
    if not os.path.exists(token_map_path):
        cache_dir = snapshot_download(
            os.path.dirname(token_map_path),
            ignore_patterns=["*.bin", "*.safetensors"],
        )
        token_map_path = os.path.join(cache_dir, os.path.basename(token_map_path))
    hot_token_id = torch.load(token_map_path)
    return torch.tensor(hot_token_id, dtype=torch.int32)


token_map = load_token_map("thunlp/LLaMA3-Instruct-8B-FR-Spec/freq_32768.pt")

Fetching 4 files: 100%|██████████| 4/4 [00:05<00:00,  1.37s/it]
/tmp/ipykernel_1872317/145328069.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  hot_token_id = torch.lo

In [16]:
torch.set_printoptions(edgeitems=10)
print(token_map[-200:])
print(token_map.shape)

tensor([ 93622,  93646,  93683,  93824,  93828,  93830,  93988,  93999,  94071,
         94082,  ..., 122098, 123282, 124232, 125341, 126029, 126058, 126437,
        126459, 128000, 128009], dtype=torch.int32)
torch.Size([32768])


In [5]:
def load_token_map_from_model(repo_id: str, file_name: str, attribute: str):
    cache_dir = snapshot_download(
        repo_id,
        ignore_patterns=["*.safetensors"],  # Don't exclude .bin files
    )
    model_path = os.path.join(cache_dir, file_name)
    
    # Load the model file
    model_data = torch.load(model_path)
    
    # Extract the requested attribute
    if attribute in model_data:
        token_data = model_data[attribute]
        return torch.tensor(token_data, dtype=torch.int32)
    else:
        raise ValueError(f"Attribute '{attribute}' not found in the model file")

# Load the d2t attribute from the specified model
token_map2 = load_token_map_from_model(
    "jamesliu1/sglang-EAGLE3-Llama-3.1-Instruct-8B", 
    "pytorch_model.bin", 
    "d2t"
)

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 14.32it/s]
/tmp/ipykernel_1872317/2522272248.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_data = torch.load

In [19]:
print(token_map2[:200])
print(token_map2 + torch.arange(token_map2.shape[0]))
print(token_map2.dtype)

tensor([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  ..., 145, 145, 145,
        145, 145, 145, 145, 145, 145, 145], dtype=torch.int32)
tensor([     0,      1,      2,      3,      4,      5,      6,      7,      8,
             9,  ..., 114366, 114405, 118792, 122371, 123537, 124085, 125800,
        126437, 127196, 128009])
torch.int32


In [15]:
test = load_token_map_from_model(
    "jamesliu1/sglang-EAGLE3-Llama-3.1-Instruct-8B", 
    "pytorch_model.bin", 
    "t2d"
)

print(len(test))

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 21454.24it/s]
/tmp/ipykernel_1872317/2522272248.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_data = torch.l

128256


/tmp/ipykernel_1872317/2522272248.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(token_data, dtype=torch.int32)
